# Лабораторная работа №2. Обеспечение качества данных
**Шашков Глеб, поток 3.1** · Визуализация и моделирование

Цель — исследовать качество исходных данных о вакансиях Monster.com, привести к единому формату адреса, зарплаты и занятость. Значения не исправляются без оснований; сомнительные случаи выделяются отдельно. Это самостоятельная реализация задания.


## 0. Загрузка данных
Набор содержит вакансии из США. Перед анализом проверяем наличие исходного CSV. Если его нет, пробуем скачать публичную копию датасета; в Google Colab предусмотрена ручная загрузка.


In [ ]:
import re
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 25)
pd.set_option("display.max_colwidth", 75)

names = ["monster_com-job_sample.csv", "monster_com_job_sample.csv"]
dataset = next((Path(name) for name in names if Path(name).exists()), None)

if dataset is None:
    try:
        link = "https://huggingface.co/datasets/jason1966/PromptCloudHQ_us-jobs-on-monstercom/resolve/main/monster_com-job_sample.csv"
        print("Скачивание открытой копии Monster.com...")
        urlretrieve(link, names[0])
        dataset = Path(names[0])
    except Exception as error:
        print("Автоматическое скачивание недоступно:", str(error)[:150])
        try:
            from google.colab import files
            uploaded = files.upload()
            dataset = next((Path(name) for name in uploaded if name.lower().endswith(".csv")), None)
        except ImportError:
            pass

if dataset is None:
    raise FileNotFoundError("Нужно добавить monster_com-job_sample.csv в среду ноутбука.")

jobs = pd.read_csv(dataset, low_memory=False)
original_columns = jobs.columns.tolist()
print("Загружен файл:", dataset)
print("Строк:", len(jobs), "| столбцов:", len(jobs.columns))
display(jobs.sample(min(5, len(jobs)), random_state=31))


## 1. Первичная проверка качества
Проверяем типы полей, пропуски, признаки-константы, пересечения значений и потенциальные дубли. Сначала считаем, а затем выбираем, какие преобразования действительно нужны.


In [ ]:
overview = pd.DataFrame({
    "Тип": jobs.dtypes.astype(str),
    "Всего записей": len(jobs),
    "Пропусков": jobs.isna().sum(),
    "Доля пропусков, %": (jobs.isna().mean() * 100).round(2),
    "Различных значений": jobs.nunique(dropna=False)
}).sort_values("Пропусков", ascending=False)

display(overview)
too_empty = jobs.isna().mean(axis=1).gt(0.75)
print("Строк с более чем 75% пустых полей:", too_empty.sum())
print("После удаления ВСЕХ строк с пропусками осталось бы:", len(jobs.dropna()))

fig, ax = plt.subplots(figsize=(10, 5))
overview["Пропусков"].sort_values().plot(kind="barh", color="#49719e", ax=ax)
ax.set(title="Пропуски по полям", xlabel="Число пустых значений", ylabel="")
plt.tight_layout()
plt.show()


### Что показывают первые результаты
В исходном наборе есть столбцы с одинаковыми значениями, а в некоторых текстовых полях информация записана непоследовательно. Удалять все неполные строки обычно нерационально: мы теряем полезные вакансии. Для адресов, зарплат и видов занятости создаём дополнительные поля, сохраняя исходный текст.


In [ ]:
constants = overview.index[overview["Различных значений"].eq(1)].tolist()
print("Константные признаки:", constants)

sector = jobs["sector"].astype("string").str.strip().str.casefold()
company = jobs["organization"].astype("string").str.strip().str.casefold()
same_value = (sector == company).fillna(False) & sector.notna()
print("Совпадения organization и sector:", int(same_value.sum()))
display(jobs.loc[same_value, ["organization", "sector"]].head(6))

identifiers = [col for col in jobs.columns
               if jobs[col].notna().all() and jobs[col].nunique(dropna=False) == len(jobs)]
other_fields = [col for col in jobs.columns if col not in identifiers]
duplicates = jobs.duplicated(subset=other_fields, keep=False) if other_fields else pd.Series(False, index=jobs.index)
print("Полностью уникальные столбцы:", identifiers)
print("Строки, попавшие в группы повторов без уникальных полей:", int(duplicates.sum()))
print("Повторяющиеся uniq_id:", int(jobs["uniq_id"].duplicated().sum()))
display(jobs.loc[duplicates, ["job_title", "location", "uniq_id"]].head(5))


## 2. Задание 1 — адреса вакансий
Одна и та же география встречается в разных формах: город + штат + индекс, город + штат, только индекс и другие комбинации. Мы сначала распознаём компоненты по строгим шаблонам, затем отдельно разбираем нераспознанные адреса. Ошибочные значения не подменяем угаданными.


In [ ]:
states = set(("AL AK AZ AR CA CO CT DE FL GA HI ID IL IN IA KS KY LA "
              "ME MD MA MI MN MS MO MT NE NV NH NJ NM NY NC ND OH OK OR "
              "PA RI SC SD TN TX UT VT VA WA WV WI WY DC").split())
zip_pattern = re.compile(r"(?<!\d)\d{5}(?:-\d{4})?(?!\d)")
state_pattern = re.compile(r"\b[A-Z]{2}\b")
city_pattern = re.compile(r"^[A-Za-z][A-Za-z .'\-]{0,65}$")
non_places = {"remote", "anywhere", "n/a", "unknown", "us", "usa", "united states"}

def parse_place(value):
    if pd.isna(value) or not str(value).strip():
        return pd.Series([None, None, None, "пусто"])

    text = re.sub(r"\s+", " ", str(value)).strip()
    if text.lower() in non_places or len(text) > 100:
        return pd.Series([None, None, None, "не распознано"])

    zip_matches = list(zip_pattern.finditer(text))
    if len(zip_matches) > 1:
        return pd.Series([None, None, None, "не распознано"])
    postal = zip_matches[0].group() if zip_matches else None
    remainder = zip_pattern.sub(" ", text, count=1) if postal else text

    state_matches = [match for match in state_pattern.finditer(remainder) if match.group() in states]
    if len(state_matches) > 1:
        return pd.Series([None, None, None, "не распознано"])
    region = state_matches[0].group() if state_matches else None
    if state_matches:
        match = state_matches[0]
        remainder = remainder[:match.start()] + " " + remainder[match.end():]

    city = re.sub(r"\s+", " ", remainder.strip(" ,;/|-")).strip() or None
    if city and (not city_pattern.fullmatch(city) or len(city.split()) > 6
                 or city.casefold() in non_places):
        return pd.Series([None, None, None, "не распознано"])
    if not any((city, region, postal)):
        return pd.Series([None, None, None, "не распознано"])

    parts = [name for name, found in (("город", city), ("штат", region), ("индекс", postal)) if found]
    return pd.Series([city, region, postal, " + ".join(parts)])

places = jobs["location"].apply(parse_place)
places.columns = ["city", "state_code", "zip_code", "location_format"]

distribution = places["location_format"].value_counts()
display(distribution.rename("Количество").to_frame())
distribution.plot(kind="bar", figsize=(10, 4), color="#477e9f", title="Форматы адресов")
plt.ylabel("Вакансий")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()
print("Наиболее частый формат:", distribution.index[0])


### Задания 1.2 и 1.3: нераспознанные адреса
Проверяем, какие элементы можно безопасно восстановить из сложных строк. Например, если полный адрес ошибочен, но код штата однозначно указан, его можно сохранить отдельно. Остальное помечаем для ручной проверки.


In [ ]:
bad_locations = jobs.loc[places["location_format"].isin(["не распознано", "пусто"])].copy()
display(bad_locations[["job_title", "location"]].head(12))
print("Первоначально не распознано:", len(bad_locations))

def salvage_place(value):
    if pd.isna(value):
        return (None, None)
    text = str(value)
    codes = [match for match in state_pattern.findall(text) if match in states]
    zips = zip_pattern.findall(text)
    region = codes[0] if len(codes) == 1 else None
    postal = zips[0] if len(zips) == 1 else None
    return (region, postal)

needs_review = places["location_format"].eq("не распознано")
recovered = jobs.loc[needs_review, "location"].apply(salvage_place)
recovered = pd.DataFrame(recovered.tolist(), index=recovered.index, columns=["recovered_state", "recovered_zip"])

for field, backup in [("state_code", "recovered_state"), ("zip_code", "recovered_zip")]:
    places.loc[needs_review, field] = places.loc[needs_review, field].fillna(recovered[backup])

places["location_requires_review"] = needs_review
print("Частично восстановлены штат и/или индекс:",
      int(recovered.notna().any(axis=1).sum()))
display(pd.concat([jobs["location"], places], axis=1).loc[needs_review].head(10))


## 3. Задание 2 — зарплаты
Значения смешивают минимум, максимум и период: час, неделя, месяц, год. В отдельные поля выделяем числовые границы и единицу времени. Для одного указанного значения по условию работы считаем сумму минимальной, даже если в тексте написано Up to.


In [ ]:
money_pattern = re.compile(r"(?<!\w)\$?\s*(\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+(?:\.\d+)?)\s*([kK])?(?!\w)")
period_patterns = {
    "hour": re.compile(r"(?:/|per\s+)\s*hours?\b|\bhourly\b", re.I),
    "week": re.compile(r"(?:/|per\s+)\s*weeks?\b|\bweekly\b", re.I),
    "month": re.compile(r"(?:/|per\s+)\s*months?\b|\bmonthly\b", re.I),
    "year": re.compile(r"(?:/|per\s+)\s*years?\b|\bannually\b|\byearly\b|\bannual\b", re.I)
}

def parse_pay(value):
    if pd.isna(value) or not str(value).strip():
        return pd.Series([np.nan, np.nan, None, "пусто", False])
    text = re.sub(r"\s+", " ", str(value)).strip()
    amounts = []
    for match in money_pattern.finditer(text):
        amount = float(match.group(1).replace(",", ""))
        if match.group(2):
            amount *= 1000
        amounts.append(amount)
    period = next((name for name, pattern in period_patterns.items() if pattern.search(text)), None)

    if not amounts:
        return pd.Series([np.nan, np.nan, period, "без числовой суммы", False])

    if len(amounts) > 1:
        low, high = amounts[:2]
        order_error = low > high
        category = "диапазон"
    else:
        low, high, order_error = amounts[0], np.nan, False
        if re.search(r"\bup to\b|\bдо\b", text, flags=re.I):
            category = "одна сумма (up to)"
        elif "+" in text:
            category = "одна сумма (+)"
        else:
            category = "одна сумма"
    return pd.Series([float(low), float(high), period, category, order_error])

pay = jobs["salary"].apply(parse_pay)
pay.columns = ["salary_min", "salary_max", "salary_period", "salary_format", "salary_range_error"]
display(pay["salary_format"].value_counts().rename_axis("Формат").to_frame("Количество"))
pay["salary_format"].value_counts().plot.bar(
    figsize=(10, 4), color="#598c8c", title="Как записана зарплата"
)
plt.ylabel("Количество")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()
print("Обратные диапазоны, требующие проверки:", int(pay["salary_range_error"].sum()))
display(pd.concat([jobs["salary"], pay], axis=1).dropna(subset=["salary_min"]).head(8))


### Задание 2.2: средняя и медианная зарплата
Сравниваем только значения, в которых период указан явно. Средняя чувствительнее к выбросам; медиана обычно лучше отражает типичное предложение.


In [ ]:
pay_known = pay.dropna(subset=["salary_min", "salary_period"])
salary_stats = (pay_known.groupby("salary_period")["salary_min"]
                .agg(count="count", average="mean", median="median")
                .round(2))
display(salary_stats)
if not salary_stats.empty:
    salary_stats[["average", "median"]].plot(
        kind="bar", figsize=(9, 4), rot=0, color=["#4b7296", "#d49a5d"],
        title="Средняя и медианная минимальная зарплата"
    )
    plt.xlabel("Период")
    plt.ylabel("Сумма, USD")
    plt.tight_layout()
    plt.show()


### Задание 2.3: восстановление периода
У каждой группы есть свой масштаб сумм, но отдельные диапазоны могут пересекаться. Сначала оцениваем распределения и отсеиваем выбросы; затем сравниваем неизвестную сумму с усечёнными средними. Если подходят сразу несколько периодов и разница недостаточно велика, не делаем предположений.


In [ ]:
period_profiles = {}
for period, group in pay_known.groupby("salary_period"):
    values = group["salary_min"].dropna().astype(float)
    if len(values) < 5:
        continue
    q05, q25, q75, q95 = values.quantile([.05, .25, .75, .95]).tolist()
    middle = values.between(q05, q95)
    mean_robust = values[middle].mean()
    period_profiles[period] = {"low": q05, "high": q95, "average": mean_robust}
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.hist(values[middle], bins=25, color="#5782a2", edgecolor="white")
    ax.set(title=f"Зарплаты за {period}: центральные 90% наблюдений",
           xlabel="Минимальная сумма, USD", ylabel="Количество")
    plt.tight_layout()
    plt.show()

display(pd.DataFrame(period_profiles).T.round(2))

def estimate_period(amount):
    if pd.isna(amount) or amount <= 0:
        return None
    candidates = []
    for name, profile in period_profiles.items():
        if profile["low"] <= amount <= profile["high"] and profile["average"] > 0:
            distance = abs(np.log(amount / profile["average"]))
            candidates.append((distance, name))
    candidates.sort()
    if len(candidates) == 1:
        return candidates[0][1]
    if len(candidates) >= 2 and candidates[1][0] - candidates[0][0] >= np.log(1.4):
        return candidates[0][1]
    return None

pay["salary_period_estimated"] = pd.Series([None] * len(pay), index=pay.index, dtype="object")
unknown_period = pay["salary_min"].notna() & pay["salary_period"].isna()
pay.loc[unknown_period, "salary_period_estimated"] = (
    pay.loc[unknown_period, "salary_min"].apply(estimate_period)
)
pay["salary_period_final"] = pay["salary_period"].combine_first(pay["salary_period_estimated"])
pay["salary_period_origin"] = np.where(
    pay["salary_period"].notna(), "исходный период",
    np.where(pay["salary_period_estimated"].notna(), "оценка по сумме", "не определён")
)
print("Зарплаты без периода до анализа:", int(unknown_period.sum()))
print("Оценка периода получена:", int(pay["salary_period_estimated"].notna().sum()))
print("Остались без уверенного периода:",
      int((pay["salary_min"].notna() & pay["salary_period_final"].isna()).sum()))


## 4. Задание 3 — тип занятости
В job_type смешаны график и оформление отношений с работодателем. Разбираем их по отдельности, затем применяем значения по умолчанию Full Time и Employee. Для явно указанных стажировок, контрактов и сезонных работ не подменяем тип Employee.


In [ ]:
display(jobs["job_type"].value_counts(dropna=False).head(35).rename("Количество").to_frame())

def parse_employment(value):
    text = "" if pd.isna(value) else re.sub(r"\s+", " ", str(value)).strip().lower()

    if re.search(r"\bpart[\s-]*time\b", text):
        schedule = "Part Time"
    elif re.search(r"\bper[\s-]*diem\b", text):
        schedule = "Per Diem"
    else:
        schedule = "Full Time"

    if re.search(r"\bintern(?:ship)?\b", text):
        category = "Intern"
    elif re.search(r"\bcontract\b|\btemporary\b|\bproject\b", text):
        category = "Contract"
    elif re.search(r"\bseasonal\b", text):
        category = "Seasonal"
    elif re.search(r"\bexempt\b", text) and not re.search(r"\bemployee\b", text):
        category = "Other"
    else:
        category = "Employee"
    return pd.Series([schedule, category])

employment = jobs["job_type"].apply(parse_employment)
employment.columns = ["work_schedule", "employment_type"]
display(pd.concat([jobs["job_type"], employment], axis=1).drop_duplicates().head(30))

employment_counts = pd.crosstab(employment["work_schedule"], employment["employment_type"])
display(employment_counts)
employment_counts.plot(kind="bar", stacked=True, figsize=(10, 5), rot=0,
                       title="Тип занятости по полноте рабочего времени")
plt.ylabel("Число вакансий")
plt.xlabel("График работы")
plt.legend(title="Тип", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 5. Итог, проверки и экспорт
Добавляем новые признаки к копии исходной таблицы. Удалений строк нет; исходные 14 полей остаются без изменений. После экспорта проверяем сохранённый CSV и показываем, какие записи требуют дополнительной проверки.


In [ ]:
result = jobs.copy()
for frame in (places, pay, employment):
    for field in frame.columns:
        result[field] = frame[field]

assert len(result) == len(jobs), "Изменилось количество строк"
assert result[original_columns].equals(jobs), "Изменились исходные данные"
assert not result.columns.duplicated().any(), "Повторились имена столбцов"

output_name = "VIM_Lab2_Shashkov_Gleb_3_1_cleaned.csv"
result.to_csv(output_name, index=False)
print("Файл сохранён:", output_name)
print("Итоговый размер:", result.shape)
print("Исходных полей:", len(original_columns))
print("Адресов для проверки:", int(result["location_requires_review"].sum()))
print("Ошибочных диапазонов зарплат:", int(result["salary_range_error"].sum()))

verify = pd.read_csv(output_name, low_memory=False)
assert len(verify) == len(result)
print("Проверка экспорта пройдена. Скачайте CSV при необходимости.")


## Основные выводы
1. Удаление всех строк с пустыми значениями может привести к потере почти всей выборки; полезнее заполнять или структурировать только нужные признаки.
2. Регулярные выражения помогают отделить город, штат и почтовый индекс, но нераспознанные адреса нужно проверять отдельно.
3. Зарплаты несопоставимы без учёта периода; восстановление периода по сумме остаётся оценкой, а не установленным фактом.
4. Виды занятости необходимо разделять на график работы и тип отношений; стажировки и контракты нельзя без оснований считать обычным наймом.

Для защиты нужно уметь объяснить правила каждой функции, причины сохранения неопределённых значений и ограничения метода.
